# Disagreement-Aware Probabilistic U-Net on LIDC

This notebook is the single-file Colab entry point for the project. It runs the project in order:

1. clone/pull the project branch and install dependencies;
2. download and validate the preprocessed LIDC crops;
3. train the three ablation variants;
4. evaluate checkpoints on the test split;
5. generate qualitative comparison figures.

The default `RUN_MODE = "smoke"` is intentionally short so `Runtime -> Run all` completes quickly and proves the full pipeline works. For final project numbers, change `RUN_MODE` to `"final"` in the configuration cell before running.

## 1. Setup

In Colab this cell clones the published `disagreement_probunet` branch. If the notebook is run from inside an existing checkout, it uses the current directory instead.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

REPO_URL = "https://github.com/idanSmoller/medical_imaging_project.git"
BRANCH = "disagreement_probunet"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_DIR = Path("/content/medical_imaging_project")
    if PROJECT_DIR.exists():
        subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(PROJECT_DIR), "switch", BRANCH], check=True)
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    PROJECT_DIR = Path.cwd()

os.chdir(PROJECT_DIR)
print("Project directory:", PROJECT_DIR)
print("Branch:")
subprocess.run(["git", "branch", "--show-current"], check=True)

In [ ]:
# Install project dependencies and the local package.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
except subprocess.CalledProcessError:
    if IN_COLAB:
        raise
    # Local fallback for locked-down machines: imports work directly from the checkout.
    sys.path.insert(0, str(PROJECT_DIR))
    sys.path.insert(0, str(PROJECT_DIR / "scripts"))
    print("Editable install failed locally; using repo paths on sys.path instead.")


## 2. Configuration

Use `RUN_MODE = "smoke"` for a quick end-to-end check, `"pilot"` for a meaningful but still limited run, and `"final"` for the full paper-style budget.

Important: the final mode is expensive. It trains 3 variants for 240000 optimizer steps each unless checkpoints already exist and can be resumed/skipped.

In [ ]:
import torch

RUN_MODE = "smoke"  # one of: "smoke", "pilot", "final"
FORCE_RETRAIN = False
EVAL_SPLIT = "test"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODE_CONFIGS = {
    "smoke": {
        "steps": 2,
        "max_train": 8,
        "max_val": 4,
        "batch_size": 1,
        "eval_batch_size": 1,
        "feature_maps": 2,
        "latent_size": 2,
        "depth": 2,
        "train_samples": 2,
        "eval_samples": 2,
        "eval_every": 1,
        "save_every": 1,
        "figure_cases": 2,
        "figure_samples": 2,
    },
    "pilot": {
        "steps": 10000,
        "max_train": None,
        "max_val": 256,
        "batch_size": 16,
        "eval_batch_size": 8,
        "feature_maps": 16,
        "latent_size": 6,
        "depth": 4,
        "train_samples": 4,
        "eval_samples": 16,
        "eval_every": 1000,
        "save_every": 1000,
        "figure_cases": 8,
        "figure_samples": 4,
    },
    "final": {
        "steps": 240000,
        "max_train": None,
        "max_val": None,
        "batch_size": 32,
        "eval_batch_size": 8,
        "feature_maps": 32,
        "latent_size": 6,
        "depth": 5,
        "train_samples": 4,
        "eval_samples": 32,
        "eval_every": 1000,
        "save_every": 1000,
        "figure_cases": 8,
        "figure_samples": 4,
    },
}

CFG = MODE_CONFIGS[RUN_MODE]
print("RUN_MODE:", RUN_MODE)
print("DEVICE:", DEVICE)
print(CFG)

## 3. Data

Download and validate DeepMind's preprocessed LIDC crops. This is about 215 MB and is reused if already present.

In [ ]:
subprocess.run([sys.executable, "scripts/download_lidc.py"], check=True)

## 4. Quick Disagreement Sanity Check

This visualizes one LIDC example, the four grader masks, and the entropy-based human disagreement map used as supervision.

In [ ]:
import matplotlib.pyplot as plt
from probunet.lidc import LIDCCrops
from probunet.disagreement import compute_disagreement

ds_preview = LIDCCrops(split="val", train=False, single_random_grader=False)
sample = ds_preview[0]
d_map = compute_disagreement(sample["masks"][None]).numpy()[0, 0]

fig, axes = plt.subplots(1, 6, figsize=(14, 2.4))
panels = [sample["image"][0].numpy()] + [sample["masks"][i].numpy() for i in range(4)] + [d_map]
titles = ["image", "mask 1", "mask 2", "mask 3", "mask 4", "human D"]
cmaps = ["gray"] * 5 + ["magma"]
for ax, panel, title, cmap in zip(axes, panels, titles, cmaps):
    ax.imshow(panel, cmap=cmap, vmin=0 if cmap == "magma" else None, vmax=1 if cmap == "magma" else None)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Train Ablations

This trains the required variants:

- `baseline`: original Probabilistic U-Net objective;
- `head`: baseline plus supervised human-disagreement prediction;
- `full`: baseline plus disagreement prediction and sample-diversity alignment.

The cell automatically resumes from `latest_checkpoint.pt` when available. If a checkpoint is already at the requested global step, it skips that variant.

In [ ]:
import shlex

VARIANT_SETTINGS = {
    "baseline": {},
    "head": {"lambda_disagreement": 0.5},
    "full": {"lambda_disagreement": 0.5, "lambda_alignment": 0.5},
}


def checkpoint_step(path):
    if not Path(path).exists():
        return None
    try:
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(path, map_location="cpu")
    return int(checkpoint.get("step", 0))


def add_optional_arg(cmd, name, value):
    if value is None:
        return
    cmd += ["--" + name.replace("_", "-"), str(value)]


def train_variant(variant):
    out_dir = Path("outputs") / "lidc_ablation" / variant
    latest = out_dir / "latest_checkpoint.pt"
    existing_step = checkpoint_step(latest)
    target_steps = CFG["steps"]

    if existing_step is not None and existing_step >= target_steps and not FORCE_RETRAIN:
        print(f"{variant}: latest checkpoint is already at step {existing_step}; skipping")
        return

    cmd = [
        sys.executable,
        "scripts/train_lidc_ablation.py",
        "--variant", variant,
        "--steps", str(target_steps),
        "--batch-size", str(CFG["batch_size"]),
        "--eval-batch-size", str(CFG["eval_batch_size"]),
        "--feature-maps", str(CFG["feature_maps"]),
        "--latent-size", str(CFG["latent_size"]),
        "--depth", str(CFG["depth"]),
        "--train-samples", str(CFG["train_samples"]),
        "--eval-samples", str(CFG["eval_samples"]),
        "--eval-every", str(CFG["eval_every"]),
        "--save-every", str(CFG["save_every"]),
        "--device", DEVICE,
        "--out-dir", str(out_dir),
    ]
    add_optional_arg(cmd, "max_train", CFG["max_train"])
    add_optional_arg(cmd, "max_val", CFG["max_val"])

    for key, value in VARIANT_SETTINGS[variant].items():
        add_optional_arg(cmd, key, value)

    if latest.exists() and not FORCE_RETRAIN:
        cmd += ["--resume", str(latest)]

    print("Running:", " ".join(shlex.quote(part) for part in cmd))
    subprocess.run(cmd, check=True)


for variant in ["baseline", "head", "full"]:
    train_variant(variant)

## 6. Final Test Evaluation

This evaluates the saved checkpoints on the test split and writes a compact CSV table. In smoke/pilot mode, the test set can be capped through the same `max_val` setting to keep runtime reasonable.

In [ ]:
from argparse import Namespace
import csv
import numpy as np
import sys

sys.path.insert(0, str(PROJECT_DIR / "scripts"))
from train_lidc_ablation import evaluate, make_loaders, make_model  # noqa: E402


def best_or_latest(variant):
    base = Path("outputs") / "lidc_ablation" / variant
    best = base / "best_checkpoint.pt"
    latest = base / "latest_checkpoint.pt"
    return best if best.exists() else latest


def load_checkpoint(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)


def evaluate_variant(variant):
    ckpt_path = best_or_latest(variant)
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Missing checkpoint for {variant}: {ckpt_path}")
    checkpoint = load_checkpoint(ckpt_path)
    saved_args = dict(checkpoint.get("args") or {})
    saved_args.update({
        "variant": variant,
        "device": DEVICE,
        "eval_split": EVAL_SPLIT,
        "eval_samples": CFG["eval_samples"],
        "eval_batch_size": CFG["eval_batch_size"],
        "max_val": CFG["max_val"],
        "num_workers": 0,
    })
    args = Namespace(**saved_args)
    _, loader = make_loaders(args)
    model = make_model(args)
    model.load_state_dict(checkpoint["model_state_dict"])
    metrics = evaluate(model, loader, args)
    metrics = {"variant": variant, "checkpoint": str(ckpt_path), **metrics}
    return metrics

results = [evaluate_variant(variant) for variant in ["baseline", "head", "full"]]
for row in results:
    print(row)

out_path = Path("outputs") / "lidc_ablation" / f"{EVAL_SPLIT}_metrics_{RUN_MODE}.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
fieldnames = sorted({key for row in results for key in row.keys()})
with open(out_path, "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)
print("Wrote", out_path)

## 7. Qualitative Figures

Generate Phase 10 comparison grids. Each figure includes the input image, four human annotations, human disagreement, baseline samples and uncertainty, full-model samples and uncertainty, predicted human disagreement, and error maps.

In [ ]:
fig_out_dir = Path("outputs") / "lidc_figures" / RUN_MODE
fig_out_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "scripts/make_lidc_figures.py",
    "--baseline-checkpoint", str(best_or_latest("baseline")),
    "--full-checkpoint", str(best_or_latest("full")),
    "--split", EVAL_SPLIT,
    "--num-cases", str(CFG["figure_cases"]),
    "--samples", str(CFG["figure_samples"]),
    "--device", DEVICE,
    "--out-dir", str(fig_out_dir),
]
print("Running:", " ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)

## 8. Display Generated Figures

In [ ]:
from IPython.display import Image, display

figure_paths = sorted(fig_out_dir.glob("case_*.png"))
print(f"Generated {len(figure_paths)} figures in {fig_out_dir}")
for path in figure_paths[: min(4, len(figure_paths))]:
    print(path)
    display(Image(filename=str(path)))

## 9. Outputs To Submit Or Report

The notebook generates these main artifacts:

- `outputs/lidc_ablation/<variant>/history.csv`: validation metrics over training;
- `outputs/lidc_ablation/<variant>/best_checkpoint.pt`: best checkpoint by validation GED;
- `outputs/lidc_ablation/<split>_metrics_<mode>.csv`: final evaluation table;
- `outputs/lidc_figures/<mode>/case_*.png`: qualitative comparison figures.

For the final report, use `RUN_MODE = "final"` or clearly label pilot/smoke results as non-final.